### **Layer specific dendritic spine dynamics in primary motor cortex (M1) during learning**
**Nikhita Kaushik |  SURFiN Fellow @ Komiyama Lab, 8/25-5/26 | Mentor: Jennifer Li** 

---

#### **Project Overview**
Dendritic spines undergo structural changes during motor learning, but whether these changes differ across cortical layers remains unclear. Here, we used longitudinal two-photon microscopy to track individual spines on L2/3 and L5 dendrites in primary motor cortex (M1) across 14 consecutive days of training & imaging, comparing structural spine dynamics between layers. 

#### **Dataset**
- **Mice:** Wild-type mice (P60–P90), n=3
- **Cortical layers:** L2/3 & L5
- **Imaging duration:** 14 days, with water-restricted motor training
- **Imaging modality:** Two-photon microscopy at 1000 nm; red + green channels acquired simultaneously
- **Volumes acquired:** 1 µm steps, 20–60 µm depth
- **Viral labeling:** L2/3: CamKII-Cre + FLEX-tdTomato (200–300 µm depth); L5 corticospinal neurons: retrograde Cre + FLEX-eGFP (injected C4–C6)
- **Spine annotation:** Custom GUI; volume measurements validated with correlated electron microscopy

#### **Experimental Timeline**
- **Week 0** — Cranial window surgery
- **Week 2** — Begin water deprivation
- **Week 4–6** — Daily two-photon imaging + lever-press training (14 days)

#### **Analysis Pipeline**
Spines are tracked across all 14 imaging days per FOV, the following is measured:
- **Spine density** — spines per µm over time, raw & day-1 normalised
- **Spine turnover** — daily addition & elimination rates per µm
- **Spine survival** — fraction of pre-existing & newly formed spines surviving to later days
- **Spine lifetime** — distribution of how long individual spines persist
- **Spine volume change** — pairwise volume changes classified by plasticity type
- **Plasticity counts** — number of spines per plasticity category per dendrite per day
- **Volume × lifetime** — relationship between initial spine size & spine lifetime duration

All quantifications are compared between L2/3 & L5 using a mixed effects model (layer x day interaction, mouse as random effect) to control for repeated measurements within animals. 


In [ ]:
# LOAD DATA

import os 
import sys
from pathlib import Path

root_dir = Path.cwd().parent
sys.path.append(str(root_dir / "scripts"))

from Dendrite_Imaging import Dendrite_Imaging
from Dendrite_Imaging_plotter import DendriteImagingPlots

# full_analysis_dir: Path to root analysis directory. Expected structure: analysis_dir/<mouse_id>/<fov_id>/<date>/<pickle_file>
full_analyzed_dir = r"Z:\People\JenLi\Imaging_Data\L23_L5_structural_imaging\Analyzed_Data"
minimal_test_dir = os.path.join(full_analyzed_dir, "temp") # Analyze subfolder within full_analyzed_dir. Leave as just "full_analyzed_dir" if analyzing full dataset.

dend = Dendrite_Imaging(minimal_test_dir)
dend.load_grouped_data(reanalyze=True,save_grouped=False)
dend.plot = DendriteImagingPlots(dend)

In [ ]:
# SPINE DENSITY
dend.spine_density()
dend.plot.density()
dend.plot.density_norm()

In [ ]:
# SPINE TURNOVER
dend.spine_turnover()
dend.plot.add_elim_line()
dend.plot.add_elim_bar()
dend.plot.turnover()

In [ ]:
# SPINE SURVIVAL
dend.spine_survival()
dend.plot.survival()

In [ ]:
# SPINE LIFETIME

# Pre-existing spines
dend.spine_lifetime()
dend.plot.lifetime()

# Newly formed spines (days 2-4)
dend.spine_lifetime(exist_only=False, formed_days=[2, 3, 4])
dend.plot.lifetime()

In [ ]:
# SPINE VOLUME CHANGE
dend.spine_volume_change()
dend.plot.volume_change()

In [ ]:
# SPINE PLASTICITY COUNT
dend.spine_plasticity_count()
dend.plot.plasticity_events()

In [ ]:
# SPINE VOLUME V LIFETIME
dend.spine_lifetime()
dend.volume_lifetime()
dend.plot.volume_lifetime_correlation()

In [ ]:
# MIXED EFFECTS MODEL

from stats import run_mixedlm_test
import pandas as pd

dend.spine_density()
dend.spine_turnover()
dend.spine_survival()
dend.spine_volume_change()
dend.spine_lifetime()

include_day = True  # set True to run with day as covariate (other="day"), False to run without

tests = [
    (dend.density,"density", "day", "spine_density"),
    (dend.density,"density_norm", "day", "spine_density"),
    (dend.turnover,"turnover_per", "day", "turnover"),
    (dend.turnover, "elim_per", "day", "elimination"),
    (dend.turnover,"new_per", "day", "new_spines"),
    (dend.survival,"pre_frac", "day", "survival_preexisting"),
    (dend.survival,"new_frac", "day", "survival_new"),
    (dend.dend_volchange_df, "vol_change", "day2","volume_change"),
]

for df, _, _, _ in tests:
    extracted_mouse = df.groupby("dendrite_ID")["dendrite_ID"].first().str.extract(r"(JL\d+)")[0]
    df["mouse_ID"] = df["dendrite_ID"].map(extracted_mouse)


all_results = pd.concat([
    run_mixedlm_test(
        df=df,
        value_col=val,
        factor="layer",
        random_effect="mouse_ID",
        other_factors=day_col if include_day else None,
        include_interaction=True,
        alpha=0.05,
        name=name
    )
    for df, val, day_col, name in tests
], ignore_index=True)

results_summary = all_results[
    (all_results["test"] == "lrt") &
    (all_results["term"].isin(["layer", "interaction"]))
][["comparison", "term", "p", "significant"]]

display(results_summary)

In [ ]:
# EXPORT PLOTS

# dend.plot.density(savefig="plots/spine_density.png")
# dend.plot.density_norm(savefig="plots/spine_density_norm.png")
# dend.plot.add_elim_line(savefig="plots/add_elim_line.png")
# dend.plot.add_elim_bar(savefig="plots/add_elim_bar.png")
# dend.plot.turnover(savefig="plots/turnover.png")
# dend.plot.survival(savefig="plots/survival.png")
# dend.plot.volume_change(savefig="plots/volume_change.png")
# dend.plot.plasticity_events(savefig="plots/plasticity_count.png")
# dend.spine_lifetime()
# dend.plot.lifetime(savefig="plots/lifetimes.png")
# dend.spine_lifetime(exist_only=False, formed_days=[2, 3, 4])
# dend.plot.lifetime(savefig="plots/lifetimes_2_4.png")
# dend.plot.volume_lifetime_correlation(savefig="plots/vol_life_corr.png")